In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
import clickhouse_connect

# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir',
    settings={
        'max_execution_time': 90,
        'max_memory_usage': 2_000_000_000,  # 2GB
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table (correct schema)
gsma_query = """
SELECT
    tac,
    oem                    AS manufacturer,
    brand                  AS brand,
    model                  AS model_name,
    if(
        marketing_name = '' OR isNull(marketing_name),
        concat(brand, ' ', model),
        marketing_name
    )                       AS device_name,
    device_type,
    os_family              AS operating_system,
    os_version,
    year_released,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    sim_slots
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)


In [2]:
len(gsma_df)

290402

In [5]:
gsma_df.head(10)

,tac,manufacturer,brand,model_name,device_name,device_type,operating_system,os_version,year_released,has_2g,has_3g,has_4g,has_5g,sim_slots
0,35697403,Not Known,Not Known,ROWEL K658,Not Known ROWEL K658,Handheld,,,0,0,0,0,0,0
1,35697404,Not Known,G crown,"G265, G765, G865, G965","G crown G265, G765, G865, G965",Handheld,,,0,0,0,0,0,0
2,35697405,Not Known,QMobile,Q4,Q4 TV,Handheld,Other,,2013,1,0,0,0,0
3,35697406,Not Known,Apple,iPad mini (A1600),iPad mini 3,Tablet,iOS,8_1,2014,1,1,1,0,1
4,35697407,Not Known,TC,TC F6,TC TC F6,Mobile Phone/Feature phone,,,0,0,0,0,0,0
5,35697408,Not Known,Samsung,SM-A510F,Galaxy A5 (2016),Smartphone,Android,5.1.1,2016,1,1,1,0,0
6,35697409,Foxconn International Holding,NOKIA,RM-1187,216 Dual Sim,Mobile Phone/Feature phone,Other,,2016,1,0,0,0,0
7,35697410,Samsung Korea,Samsung,SM-A260F/DS,Galaxy A2 Core,Smartphone,Android,8.1,2019,1,1,1,0,2
8,35697478,Samsung Korea,Samsung,SM-S127DL,Galaxy A12,Smartphone,Android,10,2020,1,1,1,0,1
9,35697487,Seiko Solutions Inc,Seiko,TC-T300,Seiko TC-T300,Modem,,,0,0,0,0,0,0
